# GOV-02 — Public Service SLA Breach Prediction

**Reproducible demo.** Runs top-to-bottom on a clean Colab runtime: live API download →
target definition → causal workload features → training → evaluation on unseen data → inference.

The manager's question: *of the cases arriving today, which 20% should we expedite?*

## 0. Setup

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Fakhrillo/nyc311-sla-breach-prediction.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not Path("src").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, "repo"], check=True)
    os.chdir("repo/sla" if Path("repo/sla").exists() else "repo")
    subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=False)

sys.path.insert(0, str(Path.cwd() / "src"))
import sla as S
print("working dir:", Path.cwd())

## 1. Why this project has to define its own target

NYC 311 publishes a `due_date` column — the city's own service deadline. It would be the
ideal target. Before building anything on it, check how often it is actually filled in:

In [ ]:
import urllib.parse, urllib.request, json

def count(where):
    q = urllib.parse.urlencode({"$select": "count(1) as n", "$where": where})
    url = f"https://data.cityofnewyork.us/resource/erm2-nwe9.json?{q}"
    with urllib.request.urlopen(url, timeout=120) as r:
        return int(json.load(r)[0]["n"])

window = "created_date between '2024-01-01' and '2024-03-31'"
total = count(window)
with_due = count(window + " AND due_date IS NOT NULL")
print(f"Q1 2024 cases:        {total:,}")
print(f"with a due_date:      {with_due:,}  ({with_due / total:.2%})")

Under 1%. The city's SLA field is effectively abandoned in practice, so the service window
has to be defined here — which is exactly what the brief asks for, and the single decision
this project most has to justify.

## 2. Download the scoped slice

Q1 2024, the 12 request types that make up ~60% of volume. Paged by **date window**, not
`$offset`: Socrata times out on any `$order` over a range this size, and unordered offset
paging can silently repeat or skip rows. Consecutive date windows cover the period exactly once.

First run takes 2–3 minutes; after that it reads the cached CSV.

In [ ]:
df = S.build_dataset()
print(f"{len(df):,} cases | {df.created_date.min():%Y-%m-%d} -> {df.created_date.max():%Y-%m-%d}")
print(f"still open: {df.closed_date.isna().mean():.3%}")
print(df.agency.value_counts().to_string())

## 3. EDA — resolution time is wildly heterogeneous

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))

order = df.groupby("complaint_type").resolution_hours.median().sort_values()
data = [df.loc[df.complaint_type == t, "resolution_hours"].dropna() for t in order.index]
# tick labels set separately: boxplot's own label kwarg was renamed across matplotlib versions
ax[0].boxplot(data, vert=False, showfliers=False)
ax[0].set_yticks(range(1, len(order) + 1))
ax[0].set_yticklabels([t[:22] for t in order.index], fontsize=8)
ax[0].set_xscale("log")
ax[0].set(title="Resolution time by request type (log hours)", xlabel="hours")

daily = df.set_index("created_date").resample("D").agency_backlog.median()
ax[1].plot(daily.index, daily.values, color="#b4542a")
ax[1].set(title="Median agency backlog at intake", xlabel="", ylabel="open cases")
ax[1].tick_params(axis="x", rotation=45)

plt.tight_layout(); plt.show()

A noise complaint is typically closed in about an hour; a water leak takes weeks. **One global
deadline would be meaningless** — it would simply relabel the request type. Hence a window per type.

## 4. Split first, then define the windows

Order matters. The service windows are percentiles of resolution time, so fitting them on all
the data would bake the test period's outcomes into the labels the model is scored against.

In [ ]:
train, val, test = S.time_split(df)
sla = S.fit_sla(train)          # <- training period only

snapshot = df.created_date.max()
train, val, test = (S.apply_sla(p, sla, snapshot) for p in (train, val, test))

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:<6} {len(part):>7,} cases | {part.created_date.min():%Y-%m-%d} "
          f"-> {part.created_date.max():%Y-%m-%d} | breach rate {part.y.mean():.3f}")

pd.Series(sla["per_type"]).sort_values(ascending=False).round(1).rename("window (hours)").to_frame()

### The target is deliberately hard

Because the window is the 75th percentile *of each type*, every type breaches about 25% of the
time by construction. Knowing the request type therefore tells you almost nothing about the
label — the model is forced to find signal in queue state, geography, channel and timing instead.

In [ ]:
train.groupby("complaint_type").y.agg(["size", "mean"]).round(3).sort_values("mean")

## 5. Baselines, then real models

Two baselines. The prior is the no-skill floor. The **type-rate baseline** is the honest one:
it predicts each request type's historical breach rate, so beating it means finding something
beyond "what kind of case is this".

In [ ]:
import time

CONFIGS = [
    ("baseline_prior", {}),
    ("baseline_type", {}),
    ("logreg", {"C": 1.0}),
    ("hgb", {"max_iter": 200, "learning_rate": 0.1, "max_leaf_nodes": 31}),
    ("rf", {}),
]

rows = []
for name, cfg in CONFIGS:
    t0 = time.time()
    pipe = S.make_model(name, **cfg).fit(train[S.FEATURES], train.y)
    m = S.evaluate(val.y, pipe.predict_proba(val[S.FEATURES])[:, 1])
    rows.append({"model": name, **{k: m[k] for k in
                ("pr_auc", "roc_auc", "recall_at_20pct", "brier")}, "secs": round(time.time() - t0, 1)})

pd.DataFrame(rows).round(4)

## 6. Final evaluation on unseen data

The test set is touched exactly once, here.

In [ ]:
bundle = S.load_model()  # produced by `python train.py`
scores = bundle["pipeline"].predict_proba(test[S.FEATURES])[:, 1]
final = S.evaluate(test.y, scores, bundle["threshold"])

base = S.make_model("baseline_type").fit(train[S.FEATURES], train.y)
base_m = S.evaluate(test.y, base.predict_proba(test[S.FEATURES])[:, 1], bundle["threshold"])

pd.DataFrame({"model": final, "type-rate baseline": base_m}).loc[
    ["pr_auc", "roc_auc", "recall_at_20pct", "recall_at_10pct", "brier", "precision", "recall"]
].round(4)

In [ ]:
caps = np.arange(0.05, 0.55, 0.05)
plt.figure(figsize=(6, 3.8))
plt.plot(caps, [S.recall_at_capacity(test.y, scores, c) for c in caps], "o-", label="model")
plt.plot(caps, caps, "--", color="grey", label="expediting at random")
plt.xlabel("share of arrivals a supervisor can expedite"); plt.ylabel("share of breaches caught")
plt.title("Return on a fixed expediting budget"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 7. What a random split would have told us

The brief asks when the prediction is made. A random split answers "never" — it trains on March
to score January and smears the backlog features across the boundary. This is the single easiest
way to overstate a project like this one:

In [ ]:
from sklearn.model_selection import train_test_split

labelled = S.apply_sla(df, sla, snapshot)
rtr, rva = train_test_split(labelled, test_size=0.15, random_state=0, shuffle=True)
rnd = S.make_model("hgb", max_iter=200, learning_rate=0.1, max_leaf_nodes=31)
rnd.fit(rtr[S.FEATURES], rtr.y)
rnd_m = S.evaluate(rva.y, rnd.predict_proba(rva[S.FEATURES])[:, 1])

print(f"random split (wrong)        PR-AUC {rnd_m['pr_auc']:.4f}  ROC {rnd_m['roc_auc']:.4f}")
print(f"chronological (reported)    PR-AUC {final['pr_auc']:.4f}  ROC {final['roc_auc']:.4f}")
print(f"\noverstatement: {rnd_m['pr_auc'] / final['pr_auc'] - 1:.1%} on PR-AUC")

## 8. Error analysis and fairness

In [ ]:
print(Path("artifacts/error_analysis.md").read_text())

## 9. Inference — score one incoming case

In [ ]:
examples = {
    "noise call, quiet queue": dict(created_date="2024-03-20T23:10:00",
        complaint_type="Noise - Residential", agency="NYPD", borough="BROOKLYN",
        open_data_channel_type="PHONE", agency_backlog=200),
    "heat complaint, heavy backlog": dict(created_date="2024-03-20T08:00:00",
        complaint_type="HEAT/HOT WATER", agency="HPD", borough="BRONX",
        open_data_channel_type="ONLINE", agency_backlog=9000, agency_7d_volume=15000),
    "water leak, weekend 3am": dict(created_date="2024-03-23T03:00:00",
        complaint_type="WATER LEAK", agency="HPD", borough="MANHATTAN",
        open_data_channel_type="MOBILE", agency_backlog=5000),
    "request type never seen in training": dict(created_date="2024-03-20T12:00:00",
        complaint_type="Rodent Sighting", agency="DOHMH", borough="QUEENS"),
}
for label, rec in examples.items():
    print(f"{label:<38}", S.predict_one(rec, bundle))

### Invalid input is rejected, not silently scored

In [ ]:
bad = [
    ({"complaint_type": "Noise"}, "missing required fields"),
    (dict(created_date="not-a-date", complaint_type="X", agency="A", borough="B"), "bad timestamp"),
    (dict(created_date="2024-03-20", complaint_type="X", agency="A", borough="B",
          agency_backlog=-1), "negative backlog"),
    (dict(created_date="2024-03-20", complaint_type="  ", agency="A", borough="B"), "blank type"),
]
for rec, why in bad:
    try:
        S.predict_one(rec, bundle)
        print(f"NOT CAUGHT: {why}")
    except ValueError as e:
        print(f"rejected ({why}): {e}")

## 10. Limitations

- **The target is invented, not given.** "Slower than 3 in 4 cases of its kind" is a defensible
  operational definition, but it is not a legal or published deadline, and every number here is
  relative to that choice.
- **It measures speed, not quality.** A case closed fast and a case resolved well are different
  events; only the first is visible in this data.
- **Recall varies sharply by borough** (0.14 Brooklyn to 0.58 Queens). Expediting is rationed
  public attention, so that gap redistributes municipal service geographically. It must be a
  policy decision before deployment, not a side effect.
- **The model learns the queue that existed.** Where the historical queue was already biased,
  the model reproduces it and calls it a prediction.
- **One quarter, one city, twelve request types.** Q1 includes the heating season, which drives
  HPD volume; a summer quarter would look different.
- **No causal claim.** This ranks which cases will run long. It does not show that expediting
  them helps — that needs a trial of the intervention itself.